# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
%load_ext dotenv
%dotenv 

In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
import os
from glob import glob

# Write your code below.

# Part 1: Ensuring the correct path for PRICE_DATA is set
print(os.getenv("PRICE_DATA"))

# Part 2: using glob to locate all files with the parquet extension in the directory for PRICE_DATA
parquet_files = glob(
    os.path.join(
        os.getenv('PRICE_DATA'),
        "**/*.parquet"
    ),
    recursive=True
)

# Printing the list of all parquet files in the directory PRICE_DATA
print(parquet_files)


../../05_src/data/prices/
['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.1.parquet', '../../05_src/data/prices\\AC

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Starting with a dask dataframe that contains the parquet datasets requiring transformation
dd_px = dd.read_parquet(
   os.path.join(
        os.getenv('PRICE_DATA'),
        "**/*.parquet"
    ),
    recursive=True
)

dd_px.info()
dd_px.columns


<class 'dask.dataframe.dask_expr.DataFrame'>
Columns: 10 entries, Date to Year
dtypes: datetime64[ns](1), float64(6), int32(1), string(2)

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker', 'Year'],
      dtype='object')

In [ ]:
# Importing pandas so I can use it later

import pandas as pd

In [ ]:
# Making sure I use the correct column names in the new column/variable computation:
dd_px.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker', 'Year'],
      dtype='object')

In [ ]:
dd_feat = (
    dd_px
    .groupby('ticker', group_keys=False)
    .apply(
        lambda x: (
            x.sort_values('Date')
             .assign(
                 Close_lag_1=lambda df: df['Close'].shift(1),
                 Adj_Close_lag_1=lambda df: df['Adj Close'].shift(1),
                 Returns=lambda df: df['Close'] / df['Close_lag_1'] - 1,
                 hi_lo_range=lambda df: df['High'] - df['Low'],
             )
        )
    )
)
        
dd_feat
dd_feat.compute()

C:\Users\luiss\AppData\Local\Temp\ipykernel_15212\3860053454.py:4: UserWarning: `meta` is not specified, inferred from partial data.
Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result

  .apply(
c:\Users\luiss\Dropbox\DSI Machine Learning Course\production\production-env\Lib\site-packages\dask\dataframe\groupby.py:122: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)
c:\Users\luiss\Dropbox\DSI Machine Learning Course\production\production-env\Lib\site-packages\dask\dataframe\groupby.py:122: FutureWarning

,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range
0,1993-04-28,9.1250,9.3750,9.0000,9.1250,5.244570,7821000.0,RCL.csv,RCL,1993,NaN,NaN,NaN,0.375000
1,1993-04-29,9.1250,9.3125,9.1250,9.3125,5.352339,2118800.0,RCL.csv,RCL,1993,9.1250,5.244570,0.020548,0.187500
2,1993-04-30,9.3750,9.3750,9.1250,9.1875,5.280493,536800.0,RCL.csv,RCL,1993,9.3125,5.352339,-0.013423,0.250000
3,1993-05-03,9.1250,9.1875,9.0625,9.1875,5.280493,509600.0,RCL.csv,RCL,1993,9.1875,5.280493,0.000000,0.125000
4,1993-05-04,9.1875,9.1875,8.9375,9.0625,5.208652,787400.0,RCL.csv,RCL,1993,9.1875,5.280493,-0.013605,0.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10093,2020-03-26,11.9700,12.7400,11.9600,12.7000,12.700000,722200.0,ADX.csv,ADX,2020,12.0000,12.000000,0.058333,0.780000
10094,2020-03-27,12.5600,12.6100,12.2400,12.4000,12.400000,836500.0,ADX.csv,ADX,2020,12.7000,12.700000,-0.023622,0.370000
10095,2020-03-30,12.2900,12.7200,12.2800,12.7200,12.720000,533700.0,ADX.csv,ADX,2020,12.4000,12.400000,0.025807,0.440001
10096,2020-03-31,12.7300,12.8700,12.5400,12.5900,12.590000,712800.0,ADX.csv,ADX,2020,12.7200,12.720000,-0.010220,0.330000


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
dd_feat_pandas = dd_feat.compute()

dd_feat_pandas.head(10)

c:\Users\luiss\Dropbox\DSI Machine Learning Course\production\production-env\Lib\site-packages\dask\dataframe\groupby.py:122: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)
c:\Users\luiss\Dropbox\DSI Machine Learning Course\production\production-env\Lib\site-packages\dask\dataframe\groupby.py:122: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)


,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range
0,1993-04-28,9.1250,9.3750,9.0000,9.1250,5.244570,7821000.0,RCL.csv,RCL,1993,NaN,NaN,NaN,0.3750
1,1993-04-29,9.1250,9.3125,9.1250,9.3125,5.352339,2118800.0,RCL.csv,RCL,1993,9.1250,5.244570,0.020548,0.1875
2,1993-04-30,9.3750,9.3750,9.1250,9.1875,5.280493,536800.0,RCL.csv,RCL,1993,9.3125,5.352339,-0.013423,0.2500
3,1993-05-03,9.1250,9.1875,9.0625,9.1875,5.280493,509600.0,RCL.csv,RCL,1993,9.1875,5.280493,0.000000,0.1250
4,1993-05-04,9.1875,9.1875,8.9375,9.0625,5.208652,787400.0,RCL.csv,RCL,1993,9.1875,5.280493,-0.013605,0.2500
5,1993-05-05,9.0000,9.0625,8.8750,8.8750,5.100886,747200.0,RCL.csv,RCL,1993,9.0625,5.208652,-0.020690,0.1875
6,1993-05-06,8.9375,9.0000,8.8125,8.8125,5.064962,392800.0,RCL.csv,RCL,1993,8.8750,5.100886,-0.007042,0.1875
7,1993-05-07,8.8750,8.9375,8.8125,8.8750,5.100886,180000.0,RCL.csv,RCL,1993,8.8125,5.064962,0.007092,0.1250
8,1993-05-10,8.9375,9.0625,8.8750,8.8750,5.100886,269600.0,RCL.csv,RCL,1993,8.8750,5.100886,0.000000,0.1875
9,1993-05-11,8.9375,8.9375,8.8750,8.9375,5.136808,109400.0,RCL.csv,RCL,1993,8.8750,5.100886,0.007042,0.0625


In [85]:
# Adding the rolling average variable/feature

dd_feat_pandas['Returns'].rolling(10).mean()

dd_feat_pandas.groupby('ticker')['Returns'].describe()


,count,mean,std,min,25%,50%,75%,max
ticker,,,,,,,,
ACN,4704.0,0.000677,0.019146,-0.134543,-0.007753,0.000733,0.009027,0.163668
ADX,10097.0,0.000130,0.012887,-0.151832,-0.005979,0.000000,0.006667,0.155556
AHPI,7107.0,0.001380,0.061579,-0.347917,-0.017857,0.000000,0.017544,3.217391
ALDX,1489.0,0.000499,0.052833,-0.354582,-0.023077,0.000000,0.020649,0.700787
ALL,6756.0,0.000452,0.019419,-0.211786,-0.007716,0.000213,0.008227,0.216868
...,...,...,...,...,...,...,...,...
TZOO,4428.0,0.001157,0.051125,-0.517073,-0.016720,0.000000,0.015230,1.000000
VIAC,3604.0,0.000126,0.025763,-0.207572,-0.010676,0.000190,0.010859,0.307624
WORK,197.0,-0.001129,0.043498,-0.130168,-0.025276,-0.002727,0.020280,0.167840


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

It was *not* neccessary to convert to pandas because it is a time intensive process given that pandas cannot handle large datasets, uses data stored in memory, and runs sequentially without parallelization.

Dask would be been the *better* option as it can handle larger datasets in parallel structures. Dask is specifically designed for middle size data (it is not quite a big data framework but better than pandas which is for smaller data)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.